# 🌐 Colab Remote Server — Cloudflare Tunnel

Turns this Colab session into a remote **Jupyter Lab** server accessible from anywhere.
No tokens, no accounts needed — just run and copy the URL.

**Open in Colab:**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghassan-gaidi/finetune-lab/blob/main/notebooks/setup/colab-server-tunnel.ipynb)

---
## What You Get
- A browser-accessible Jupyter Lab at a `*.trycloudflare.com` URL
- Full filesystem access to the Colab VM
- GPU accelerated compute (T4 on free tier)
- Terminal, file browser, notebook editor

⚠️ The URL changes every session. The tunnel dies when Colab disconnects (~90min idle / 2h total on free tier).

## 1. Install cloudflared

In [ ]:
# Download cloudflared (Cloudflare Tunnel client)
# No auth/account needed for quick tunnels
import os
import subprocess
import sys
import time
import json

CLOUDFLARED = "/usr/local/bin/cloudflared"

if not os.path.exists(CLOUDFLARED):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
    !mv cloudflared {CLOUDFLARED}
    print("✅ cloudflared installed")
else:
    print("✅ cloudflared already installed")

## 2. Install Jupyter Lab

In [ ]:
# Colab usually has jupyter, but ensure lab is there
!pip install -q jupyterlab

## 3. Start Jupyter Lab + Cloudflare Tunnel

This runs Jupyter Lab on a custom port, then tunnels it through Cloudflare.

In [ ]:
import subprocess
import threading
import time
import re

JUPYTER_PORT = 8890
JUPYTER_TOKEN = "colab-finetune"  # change this if you want

def start_jupyter():
    """Start Jupyter Lab in background."""
    cmd = f"""jupyter lab --no-browser --port={JUPYTER_PORT} \
             --NotebookApp.token='{JUPYTER_TOKEN}' \
             --NotebookApp.allow_origin='*' \
             --NotebookApp.allow_remote_access=True"""
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    return proc

# Start Jupyter
jupyter_proc = start_jupyter()
time.sleep(3)  # Give it a moment

# Verify it's running
import requests
try:
    r = requests.get(f"http://localhost:{JUPYTER_PORT}/api", timeout=5,
                     headers={"Authorization": f"Token {JUPYTER_TOKEN}"})
    print(f"✅ Jupyter Lab running on port {JUPYTER_PORT}")
except Exception as e:
    print(f"⚠️  Jupyter may not be ready yet: {e}")

In [ ]:
# Start Cloudflare Tunnel in background
import subprocess

tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://localhost:{JUPYTER_PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for the tunnel URL to appear
import time
tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    print(line, end="")
    # Cloudflared prints:  trycloudflare.com https://xxx.trycloudflare.com
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f"\n" + "=" * 70)
    print(f"  🚀 JUPYTER LAB IS LIVE AT:")
    print(f"  {tunnel_url}")
    print(f"  TOKEN: {JUPYTER_TOKEN}")
    print(f"  Full URL: {tunnel_url}/lab?token={JUPYTER_TOKEN}")
    print(f"=" * 70)
    print(f"\n  API endpoint: {tunnel_url}/api/")
    print(f"  Terminal:     {tunnel_url}/terminals/")
    print(f"  Tree view:    {tunnel_url}/tree?token={JUPYTER_TOKEN}")
else:
    print("❌ Failed to get tunnel URL. Check cloudflared output above.")

## 4. (Optional) Install Fine-Tuning Dependencies

Run this if you want the environment already set up for fine-tuning.

In [ ]:
# Pre-install fine-tuning stack
!pip install -qU \
    torch torchvision torchaudio \
    transformers accelerate peft bitsandbytes \
    datasets trl \
    huggingface_hub \
    wandb \
    jupyterlab-vim  # optional vim bindings

---
## How to Connect

**From Browser:**
1. Copy the `tunnel_url` printed above
2. Open: `{tunnel_url}/lab?token={JUPYTER_TOKEN}`
3. You're in! Full GPU-accelerated Jupyter Lab on Colab.

**From Python (remote client):**
```python
import requests
BASE = "https://xxx.trycloudflare.com"
TOKEN = "colab-finetune"
r = requests.get(f"{BASE}/api/contents", headers={"Authorization": f"Token {TOKEN}"})
print(r.json())
```

**⚠️ Keep this cell running.** Stopping execution kills the tunnel.
If Colab disconnects, re-run cells 3-4 to get a new URL.